<span id="장치-명령-처리와-메시지-연동"></span>

# Device command handling and messaging integration

<span id="학습-목표"></span>

## Learning objectives

Separate input, state, and response so that the same command handling rules can be reused across messengers, the web, and device buses.

<span id="실행-조건"></span>

## Execution conditions

The basic activity uses Python only. Choose a real messenger or MQTT connection only after verifying the handling rules.

<span id="명령과-state-transition"></span>

## Commands and state transitions

<span id="시작-조건"></span>

### Starting conditions

The inputs are /start, /led on, /led off, /led invalid, and /unknown. The initial LED state is off.

<span id="진행-방법"></span>

### Procedure

1.  Write handle(text,state) to return the new state and a response message.
2.  Process the input list in order and record the state before and after each step.
3.  For invalid arguments and unknown commands, preserve the state and return usage instructions.

<span id="남길-결과"></span>

### Results to retain

- A 5-row state transition table and a function.

In [ ]:
def handle(text,state):
    if text=='/start':return state,'Commands: /led on, /led off'
    if text=='/led on':return True,'LED on'
    if text=='/led off':return False,'LED off'
    return state,'Usage: /led on | /led off'
state=False
for text in ['/start','/led on','/led off','/led invalid','/unknown']:
    next_state,reply=handle(text,state);print(text,state,next_state,reply);state=next_state


<span id="결과-기록자가-점검"></span>

## Record results and self-check

**Input:**

**Previous state:**

**Next state:**

**Response:**

<span id="확인할-점"></span>

### Points to check

- Can you check it through function calls without sending real messages?
- Does the LED state remain unchanged for invalid and unknown?

<span id="반복-명령과-공백"></span>

## Repeated commands and whitespace

<span id="시작-조건-1"></span>

### Starting conditions

Repeat /led on three times, and add ” /led off “ with leading and trailing spaces.

<span id="진행-방법-1"></span>

### Procedure

1.  Check whether repeated commands preserve the state.
2.  Define rules for strip and case handling, then compare with the original function.
3.  Also choose two inputs to reject, and check that the state is preserved.

<span id="남길-결과-1"></span>

### Results to retain

- A table with at least 6 inputs and 3 asserts.

In [ ]:
tests=['/led on','/led on','/led on',' /led off ','/LED ON','']
# TODO: Compare the handler with normalized input against the original.


<span id="결과-기록자가-점검-1"></span>

## Record results and self-check

**Input:**

**Normalization result:**

**State:**

**Accepted:**

<span id="확인할-점-1"></span>

### Points to check

- Did you apply consistent command normalization rules across inputs?
- Does repeated on avoid behaving like a toggle that reverses the state?

<span id="전송-방식과-장치-로직-분리"></span>

## Separate transport from device logic

Read [Week14-2_Telegram_Bot_Command_Handler.py](https://chorok-daddy.github.io/niil/courses/microprocessor/telegram-bot/Week14-2_Telegram_Bot_Command_Handler.py) and `handle` and `dispatch` in this notebook, and map each role to its function or line in the table. Mark roles absent from the original as 'None' and distinguish them from the test implementation.

Use the provided `outbox` log instead of sending real messages.

<span id="시작-조건-2"></span>

### Starting conditions

Organize inputs from the web, messengers, and MQTT as {request_id,device,command}. Do not execute the same request_id again.

<span id="진행-방법-2"></span>

### Procedure

1.  Validate the request ID, device, and command, then pass them to the common handler function.
2.  Handle the same request sent twice so that the state changes only once.
3.  Record the response and actual state, and draw the request–handling–response flow.

<span id="남길-결과-2"></span>

### Results to retain

- A role mapping table and 3 rows of simulated transmission logs.

In [ ]:
seen=set();outbox=[]
def dispatch(message):
    if message['request_id'] in seen:return 'duplicate'
    seen.add(message['request_id']);outbox.append(dict(message));return 'accepted'
message={'request_id':'r1','device':'fan','command':'on'}
print(dispatch(message),dispatch(message),outbox)
# TODO: Validate invalid device and command values before they enter seen.


<span id="결과-기록자가-점검-2"></span>

## Record results and self-check

**Role:**

**Original function and location:**

**Test substitute:**

**Observed results:**

<span id="확인할-점-2"></span>

### Points to check

- Can you test the same handling logic without a token?
- Did you check state updates and external transmission independently?